# pgvectorをベクトルデータベースとして使用し、レコードを登録するサンプル

## パッケージをインポート

In [ ]:
import psycopg2
from sentence_transformers import SentenceTransformer
import os

## 埋め込みモデル初期化

In [ ]:
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
model

## PostgreSQLへ接続

In [ ]:
db_config = {
    'user': os.environ['PGVECTOR_POSTGRES_USER'],
    'password': os.environ['PGVECTOR_POSTGRES_PASSWORD'],
    'host': 'llm-rag-examples-pgvector',
    'database': os.environ['PGVECTOR_POSTGRES_DB'],
    'port': '5432'
}

db_config

In [ ]:
conn = psycopg2.connect(**db_config)
cursor = conn.cursor()

## テーブル作成（存在しない場合）

In [ ]:
cursor.execute("""
    CREATE TABLE IF NOT EXISTS text_embeddings (
        id SERIAL PRIMARY KEY,
        text TEXT NOT NULL,
        embedding VECTOR(384)
    )
""")
conn.commit()

## テーブルのレコードを削除

In [ ]:
cursor.execute('TRUNCATE TABLE text_embeddings')
conn.commit()

## ドキュメントをテーブルにレコードとして登録

In [ ]:
texts = [
    '猫は可愛い動物です。',
    '犬は人間の親友と呼ばれています。',
    '東京は日本の首都です。'
]

In [ ]:
import pprint
for text in texts:
    embedding = model.encode([text])[0]
    pprint.pprint(f'Text: {text}')
    pprint.pprint(f'Embedding shape: {embedding.shape}')
    cursor.execute('INSERT INTO text_embeddings (text, embedding) VALUES (%s, %s)', (text, embedding.tolist()))
    print(f"Inserted: {text}")
    print()

In [ ]:
conn.commit()

## 登録されたレコードの確認

In [ ]:
cursor.execute('SELECT id, text FROM text_embeddings ORDER BY id')
records = cursor.fetchall()
for record in records:
    print(f'ID: {record[0]}, Text: {record[1]}')

In [ ]:
cursor.close()
conn.close()